In [ ]:
!pip install rdkit lightning tokenizers

In [ ]:
import os, math
import numpy as np
import pandas as pd
from tqdm import tqdm
from functools import partial
from sklearn.model_selection import train_test_split

from rdkit import DataStructs
from rdkit.Chem import MolFromSmiles, MolToSmiles, MolToInchiKey, rdFingerprintGenerator
import rdkit.rdBase as rkrb
import rdkit.RDLogger as rkl
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from torch import nn
import torch.nn.functional as F
from torch.nn import TransformerEncoderLayer, TransformerEncoder
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

from tokenizers import Tokenizer, models, trainers
from tokenizers.processors import TemplateProcessing

import lightning as L
from lightning.pytorch import loggers

tqdm.pandas()
COMP_DIR = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
TRAIN_PATH = os.path.join(COMP_DIR, 'train.parquet')
WORKING_DIR = '/kaggle/working'

rkl.logger().setLevel(rkl.ERROR)
rkrb.DisableLog("rdApp.error")

import rdkit
print('rdkit', rdkit.__version__)

In [ ]:
MAX_LEN = 128
BPE_PAD_ID, BPE_BOS_ID, BPE_EOS_ID = 0, 1, 2
VOCAB_SIZE = 512
MAX_TRAIN_SPECTRA = 200_000

NEEDED = ['ingest_lib', 'inchikey14', 'normalized_smiles', 'precursor_mz',
          'ms2_mzs', 'ms2_normalized_intensities']
casmi_all_df = pd.read_parquet(TRAIN_PATH, columns=NEEDED,
                               filters=[('ingest_lib', '!=', 'enveda-180')])

all_structs = casmi_all_df.inchikey14.unique()
train_structs, val_structs = train_test_split(all_structs, test_size=500, random_state=0)
casmi_train_df = casmi_all_df[casmi_all_df.inchikey14.isin(train_structs)].copy()
casmi_val_df = casmi_all_df[casmi_all_df.inchikey14.isin(val_structs)].copy()

if len(casmi_train_df) > MAX_TRAIN_SPECTRA:
    casmi_train_df = casmi_train_df.sample(n=MAX_TRAIN_SPECTRA, random_state=0)
casmi_val_df = casmi_val_df.drop_duplicates(subset=['inchikey14'], keep='first')
print('train:', len(casmi_train_df), 'val:', len(casmi_val_df))

def process_spectra(df):
    def _proc(row):
        order = np.argsort(row.ms2_normalized_intensities)[::-1]
        mzs = row.ms2_mzs[order][:MAX_LEN - 1]
        ints = row.ms2_normalized_intensities[order][:MAX_LEN - 1]
        return mzs, ints / ints.max()
    mzs, ints = zip(*df.progress_apply(_proc, axis=1))
    df['processed_mzs'] = mzs
    df['processed_intensities'] = ints
    return df

casmi_train_df = process_spectra(casmi_train_df)
casmi_val_df = process_spectra(casmi_val_df)

In [ ]:
def train_bpe_tokenizer(structures):
    os.environ["TOKENIZERS_PARALLELISM"] = "true"
    alphabet = list(set("".join(structures)))
    tok = Tokenizer(models.BPE())
    specials = ["<pad>", "<s>", "</s>"]
    tok.post_processor = TemplateProcessing(
        single="<s> $A </s>",
        special_tokens=[("<pad>", BPE_PAD_ID), ("<s>", BPE_BOS_ID), ("</s>", BPE_EOS_ID)],
    )
    trainer_ = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, initial_alphabet=alphabet,
                                   special_tokens=specials, show_progress=True)
    tok.train_from_iterator(structures, trainer_)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    return tok

tokenizer = train_bpe_tokenizer(list(casmi_train_df.normalized_smiles.unique()))
tokenizer.save(os.path.join(WORKING_DIR, 'tokenizer.json'))

def tokenize_df(df):
    df['bpe_tokenized_smiles'] = df.normalized_smiles.progress_apply(
        lambda x: np.array(tokenizer.encode(x).ids, dtype=int))
    return df

casmi_train_df = tokenize_df(casmi_train_df)
casmi_val_df = tokenize_df(casmi_val_df)

def decode_tokenized_smiles(tokens, tokenizer=tokenizer):
    return tokenizer.decode(tokens).replace(' ', '')

print('max token len:', casmi_train_df.bpe_tokenized_smiles.map(len).max())

In [ ]:
PRECURSOR_INTENSITY = 2.0
MZ_PAD_ID = 0
INTENSITY_PAD_ID = 0
BATCH_SIZE = 128
VAL_BATCH_SIZE = 16
NUM_WORKERS = 3

class PandasDataset(Dataset):
    def __init__(self, df, column_list):
        self.df = df[column_list]
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        return self.df.iloc[idx].to_dict()

class CASMIDataModule(L.LightningDataModule):
    def __init__(self, batch_size, val_batch_size=16):
        super().__init__()
        self.batch_size = batch_size
        self.val_batch_size = val_batch_size

    def setup(self, stage):
        cols = ['precursor_mz', 'processed_mzs', 'processed_intensities',
                'bpe_tokenized_smiles', 'normalized_smiles']
        if stage in ("fit", "validate"):
            self.train_df = PandasDataset(casmi_train_df, cols)
            self.val_df = PandasDataset(casmi_val_df, cols)

    def _loader(self, ds, stage, shuffle, batch_size):
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          collate_fn=partial(self._collator, stage=stage),
                          num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
                          pin_memory=True)

    def train_dataloader(self):
        return self._loader(self.train_df, 'fit', True, self.batch_size)

    def val_dataloader(self):
        return self._loader(self.val_df, 'validate', False, self.val_batch_size)

    def _collator(self, data, stage='fit'):
        mzs = [[row['precursor_mz']] + row['processed_mzs'].tolist() for row in data]
        ints = [[PRECURSOR_INTENSITY] + row['processed_intensities'].tolist() for row in data]
        n = max(len(x) for x in mzs)
        mzs = [x + [MZ_PAD_ID] * (n - len(x)) for x in mzs]
        ints = [x + [INTENSITY_PAD_ID] * (n - len(x)) for x in ints]
        mz_array = torch.tensor(mzs)
        batch = {
            'mzs': mz_array,
            'intensities': torch.tensor(ints),
            'attention_mask': torch.where(mz_array == MZ_PAD_ID, 0, 1),
        }
        labels = [row['bpe_tokenized_smiles'] for row in data]
        m = max(len(x) for x in labels)
        batch['structure_tokens'] = torch.tensor(
            [list(x) + [BPE_PAD_ID] * (m - len(x)) for x in labels])
        if stage == 'validate':
            batch['smiles'] = [row['normalized_smiles'] for row in data]
        return batch

datamodule = CASMIDataModule(batch_size=BATCH_SIZE, val_batch_size=VAL_BATCH_SIZE)

In [ ]:
class PeakEmbedder(nn.Module):
    def __init__(self, d_model, dropout, sin_dim=None, mz_log_lims=(-2., 3.), mz_log_power=1.0):
        super().__init__()
        sin_dim = sin_dim if sin_dim is not None else d_model
        wavelength = torch.pow(
            10,
            (mz_log_lims[1] - mz_log_lims[0]) * torch.pow(
                torch.linspace(0, 1, int(sin_dim / 2)), mz_log_power) + mz_log_lims[0])
        self._frequency = nn.Parameter(2 * np.pi / wavelength, requires_grad=False)
        self._ff_block_1 = nn.Sequential(
            nn.Linear(sin_dim, d_model), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model, d_model), nn.Dropout(dropout))
        self._ff_block_2 = nn.Sequential(
            nn.Linear(d_model + 1, d_model), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model, d_model), nn.Dropout(dropout))

    def forward(self, mz_tensor, intensity_tensor):
        omega = self._frequency.view(*(1 for _ in range(mz_tensor.ndim)), -1) * mz_tensor.unsqueeze(-1)
        mz_vecs = torch.cat([torch.sin(omega), torch.cos(omega)], dim=-1)
        mz_embeds = self._ff_block_1(mz_vecs)
        return self._ff_block_2(torch.cat([mz_embeds, intensity_tensor.unsqueeze(2)], dim=2))


def _init_layers(layers):
    for layer in layers:
        for _, mod in layer.named_modules():
            if isinstance(mod, nn.Linear):
                nn.init.xavier_uniform_(mod.weight)
                if mod.bias is not None:
                    nn.init.constant_(mod.bias, 0.0)
            elif isinstance(mod, nn.LayerNorm):
                nn.init.constant_(mod.weight, 1.0)
                if mod.bias is not None:
                    nn.init.constant_(mod.bias, 0.0)


class SpectrumEncoder(nn.Module):
    def __init__(self, embed_dim, n_heads, n_layers, dim_feedforward=None, dropout=0.1, activation='relu'):
        super().__init__()
        dim_feedforward = dim_feedforward if dim_feedforward is not None else 4 * embed_dim
        self.encoder = TransformerEncoder(
            TransformerEncoderLayer(embed_dim, n_heads, dim_feedforward=dim_feedforward,
                                    batch_first=True, dropout=dropout, activation=activation),
            n_layers)
        _init_layers(self.encoder.layers)

    def forward(self, sequence_input, attention_mask):
        return self.encoder(sequence_input, src_key_padding_mask=(attention_mask == 0))


class SmilesDecoder(nn.Module):
    def __init__(self, embed_dim, vocab_size, n_layers, n_heads,
                 pad_token_id=1, bos_token_id=0, eos_token_id=2,
                 dim_feedforward=None, dropout=0.1, activation='gelu',
                 validate_n_samples=10, predict_n_samples=25, max_len=1024):
        super().__init__()
        self.bos_token_id = bos_token_id
        self.pad_token_id = pad_token_id
        self.eos_token_id = eos_token_id
        self.embed_dim = embed_dim
        dim_feedforward = dim_feedforward if dim_feedforward is not None else 4 * embed_dim
        self.n_samples_per_stage = {'validate': validate_n_samples, 'predict': predict_n_samples}
        self.wte = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)
        self.wpe = nn.Embedding(max_len, embed_dim)
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=embed_dim, dim_feedforward=dim_feedforward,
                                       nhead=n_heads, dropout=dropout,
                                       activation=activation, batch_first=True),
            num_layers=n_layers)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)
        nn.init.zeros_(self.lm_head.weight)
        nn.init.normal_(self.wte.weight, mean=0.0, std=1.0)
        _init_layers(self.decoder.layers)

    def forward(self, idx, encoder_outputs, encoder_attention_mask, structure_tokens=None):
        softcap = 15
        L_ = idx.shape[1]
        pos = torch.arange(L_, device=idx.device)
        tgt = self.wte(idx) + self.wpe(pos)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(L_, device=idx.device)
        x = self.decoder(tgt=tgt, memory=encoder_outputs, tgt_mask=tgt_mask,
                         memory_key_padding_mask=(encoder_attention_mask == 0))
        logits = softcap * torch.tanh(self.lm_head(x) / softcap)
        logits = logits.float()
        out = {'logits': logits}
        if structure_tokens is not None:
            out['loss'] = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                structure_tokens[:, 1:].reshape(-1),
                ignore_index=self.pad_token_id)
        return out

    @torch.inference_mode()
    def generate(self, encoder_states, encoder_mask, n_samples=1, max_new_tokens=100, temperature=1.0):
        B = encoder_states.shape[0]
        device = encoder_states.device
        N = B * n_samples
        idx = torch.full((N, 1), self.bos_token_id, dtype=torch.long, device=device)
        logprob = torch.zeros(N, 1, device=device)
        encoder_states = encoder_states.repeat_interleave(n_samples, dim=0)
        encoder_mask = encoder_mask.repeat_interleave(n_samples, dim=0)
        done = torch.zeros(N, dtype=torch.bool, device=device)
        for _ in range(max_new_tokens):
            logits = self.forward(idx, encoder_states, encoder_mask)['logits'][:, -1, :]
            logp = logits.log_softmax(dim=-1)
            rescaled = (logits / temperature).log_softmax(dim=-1)
            if done.any():
                logp[done, :] = -float("inf");      logp[done, self.eos_token_id] = 0
                rescaled[done, :] = -float("inf");  rescaled[done, self.eos_token_id] = 0
            nxt = torch.multinomial(rescaled.softmax(dim=-1), num_samples=1)
            logprob = torch.cat((logprob, torch.take_along_dim(logp, nxt, dim=1)), dim=1)
            idx = torch.cat((idx, nxt), dim=1)
            done = done | (nxt.squeeze(1) == self.eos_token_id)
            if done.all():
                break
        return idx.reshape(B, n_samples, -1), logprob.sum(dim=-1).reshape(B, n_samples)

In [ ]:
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2)

def tanimoto_smiles(s1, s2):
    m1, m2 = MolFromSmiles(s1), MolFromSmiles(s2)
    if m1 is None or m2 is None:
        return None
    return DataStructs.TanimotoSimilarity(fp_gen.GetSparseFingerprint(m1),
                                          fp_gen.GetSparseFingerprint(m2))

class DeNovoLightningModel(L.LightningModule):
    def __init__(self, kwargs):
        super().__init__()
        self.peak_embedder = PeakEmbedder(**kwargs['peak_embedder'])
        self.spectrum_encoder = SpectrumEncoder(**kwargs['spectrum_encoder'])
        self.smiles_decoder = SmilesDecoder(**kwargs['smiles_decoder'])
        self.optimizer_config = kwargs['optimizer']

    def forward(self, batch, stage):
        enc = self.spectrum_encoder(
            self.peak_embedder(batch['mzs'], batch['intensities']), batch['attention_mask'])
        outputs = {}
        if stage in ('fit', 'validate'):
            outputs = self.smiles_decoder(batch['structure_tokens'], enc,
                                          batch['attention_mask'],
                                          structure_tokens=batch['structure_tokens'])
        if stage in ('validate', 'predict'):
            tokens, scores = self.smiles_decoder.generate(
                enc, batch['attention_mask'],
                n_samples=self.smiles_decoder.n_samples_per_stage[stage], temperature=1.)
            outputs['generated_tokens'] = tokens
            outputs['generated_scores'] = scores
        return outputs

    def training_step(self, batch):
        loss = self(batch, "fit")['loss']
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True,
                 batch_size=batch['mzs'].shape[0])
        return loss

    def validation_step(self, batch):
        out = self(batch, "validate")
        metrics = self.compute_val_metrics(out['generated_tokens'],
                                           out['generated_scores'], batch['smiles'])
        self.log_dict({"val_loss": out['loss'], **{f"val_{k}": v for k, v in metrics.items()}},
                      add_dataloader_idx=False, on_epoch=True,
                      batch_size=batch['mzs'].shape[0])

    def compute_val_metrics(self, generated_tokens, generated_scores, labels):
        decoded = [[decode_tokenized_smiles(t.tolist()) for t in seqs] for seqs in generated_tokens]
        order = [s.argsort(descending=True) for s in generated_scores]
        valid = [[sm[i] for i in o if sm[i] != '' and MolFromSmiles(sm[i]) is not None]
                 for sm, o in zip(decoded, order)]
        top = [v[0] if len(v) > 0 else None for v in valid]
        ikey = lambda s: MolToInchiKey(MolFromSmiles(s)).split('-')[0]
        match = [(t is not None and ikey(t) == ikey(l)) for t, l in zip(top, labels)]
        tan = [tanimoto_smiles(t, l) if t is not None else 0.0 for t, l in zip(top, labels)]
        return {'valid_smiles': float(np.mean([t is not None for t in top])),
                'tanimoto': float(np.mean(tan)),
                'exact match': float(np.mean(match))}

    def configure_optimizers(self):
        opt = AdamW(self.parameters(), lr=self.optimizer_config['lr'], weight_decay=0.01)
        total = int(self.trainer.estimated_stepping_batches)
        warm = max(1, int(0.05 * total))
        def f(s):
            if s < warm:
                return s / warm
            p = (s - warm) / max(1, total - warm)
            return 0.5 * (1 + math.cos(math.pi * p))
        return {"optimizer": opt, "lr_scheduler": {"scheduler": LambdaLR(opt, f), "interval": "step"}}

In [ ]:
EMBED_DIM, DIM_FEEDFORWARD, DROPOUT = 768, 3072, 0.1
LEARNING_RATE = 1e-4

model_params = {
    'peak_embedder': {'d_model': EMBED_DIM, 'dropout': DROPOUT},
    'spectrum_encoder': {'embed_dim': EMBED_DIM, 'n_heads': 12, 'n_layers': 6,
                         'dim_feedforward': DIM_FEEDFORWARD, 'dropout': DROPOUT,
                         'activation': 'gelu'},
    'smiles_decoder': {'embed_dim': EMBED_DIM, 'vocab_size': VOCAB_SIZE, 'n_layers': 6,
                       'n_heads': 12, 'pad_token_id': BPE_PAD_ID, 'bos_token_id': BPE_BOS_ID,
                       'eos_token_id': BPE_EOS_ID, 'dim_feedforward': DIM_FEEDFORWARD,
                       'dropout': DROPOUT, 'activation': 'gelu',
                       'validate_n_samples': 10, 'predict_n_samples': 25},
    'optimizer': {'lr': LEARNING_RATE},
}
PRECISION = 'bf16-mixed' if torch.cuda.is_bf16_supported() else '16-mixed'

# Smoke test: 1 batch train + 1 batch validate, để bắt lỗi sớm
smoke = L.Trainer(fast_dev_run=1, accelerator='auto', devices=1, precision=PRECISION,
                  logger=False, enable_checkpointing=False)
smoke.fit(DeNovoLightningModel(model_params), datamodule=datamodule)
del smoke
torch.cuda.empty_cache()
print('SMOKE TEST OK')

In [ ]:
RUN_NAME = 'denovo_fixed'
CKPT_DIR = os.path.join(WORKING_DIR, 'model_checkpoints', RUN_NAME)

epoch_ckpt = L.pytorch.callbacks.ModelCheckpoint(
    dirpath=CKPT_DIR, monitor='val_tanimoto', mode='max', every_n_epochs=1,
    save_top_k=1, save_last=True, filename="{epoch}-{step}-{val_tanimoto:.3f}")
step_ckpt = L.pytorch.callbacks.ModelCheckpoint(
    dirpath=os.path.join(CKPT_DIR, 'steps'), every_n_train_steps=300,
    save_top_k=1, filename='latest')

model = DeNovoLightningModel(model_params)
trainer = L.Trainer(
    max_epochs=3,
    max_time="00:10:00:00",
    devices=1, accelerator='auto', precision=PRECISION,
    log_every_n_steps=100, check_val_every_n_epoch=1,
    accumulate_grad_batches=3, gradient_clip_val=1.0,
    logger=[loggers.TensorBoardLogger(os.path.join(WORKING_DIR, 'lightning_logs'), name=RUN_NAME)],
    callbacks=[epoch_ckpt, step_ckpt], num_sanity_val_steps=0)

trainer.fit(model, datamodule=datamodule)

In [ ]:
import json
metrics = {k: float(v) for k, v in trainer.callback_metrics.items()}
print(metrics)
json.dump(metrics, open(os.path.join(WORKING_DIR, 'metrics.json'), 'w'), indent=2)

torch.save({'state_dict': model.state_dict(), 'model_params': model_params},
           os.path.join(WORKING_DIR, 'denovo_weights.pt'))
trainer.save_checkpoint(os.path.join(WORKING_DIR, 'final.ckpt'))
print(os.listdir(WORKING_DIR))